# EDA, construcción de series y diagnóstico de estacionariedad

**Laboratorio 1 — Series de Tiempo (CC3084)**

Pipeline en `src/lab.py`:

1. Carga y calidad de datos
2. EDA (temporal, países, regiones, vías, fronteras, tipos y cruces)
3. Siete series mensuales: total + Top 3 fronteras + vías Aérea/Terrestre/Marítima
4. Split cronológico 70/30
5. Diagnóstico de media y varianza (descomposición, log, ACF/PACF, ADF)

Salidas en `outputs/` (`tablas/`, `series/`, `figuras/`, `manifest.md`).

> **Notas:** solo `Turista` + `Excursionista`. Tras 2023, `País` puede ser agrupación de mercado. Atípicos de pandemia/estacionalidad no se eliminan.

## 1. Configuración

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import lab as L

print("ROOT:", ROOT)
print("DATA_PATH:", L.DATA_PATH)
print("TIPOS_VALIDOS:", L.TIPOS_VALIDOS)
print("TRAIN_RATIO:", L.TRAIN_RATIO)
print("FRECUENCIA:", L.FRECUENCIA)
print("PERIODO_ESTACIONAL:", L.PERIODO_ESTACIONAL)
print("OUTPUT_DIR:", L.OUTPUT_DIR)

ROOT: /home/cisco/Universidad/cuarto/segundo_semestre/data_science/lab1/DM-Laboratorio-1.-Series-de-Tiempo
DATA_PATH: /home/cisco/Universidad/cuarto/segundo_semestre/data_science/lab1/DM-Laboratorio-1.-Series-de-Tiempo/data/Base_Migracion_2009-2026jun.xlsx
TIPOS_VALIDOS: ['Turista', 'Excursionista']
TRAIN_RATIO: 0.7
FRECUENCIA: MS
PERIODO_ESTACIONAL: 12
OUTPUT_DIR: /home/cisco/Universidad/cuarto/segundo_semestre/data_science/lab1/DM-Laboratorio-1.-Series-de-Tiempo/outputs/parte1


## 2. Ejecutar pipeline

Regenera tablas, series CSV y figuras. La lógica reutilizable está en `src/lab.py` (`analizar_serie` evita repetir el diagnóstico por frontera/vía).

In [2]:
out = L.run()
print(f"Series: {out['n_series']}")
print(f"Fecha de corte (fin train): {out['fecha_corte']}")
print(f"|diff| máx vías vs total: {out['diff_vias_max']}")
print(f"Manifest: {out['manifest']}")
for r in out["resultados"]:
    print("-", r.interpretacion)

Series: 7
Fecha de corte (fin train): 2021-03-01
|diff| máx vías vs total: 5.820766091346741e-11
Manifest: /home/cisco/Universidad/cuarto/segundo_semestre/data_science/lab1/DM-Laboratorio-1.-Series-de-Tiempo/outputs/parte1/manifest.md
- Total mensual: 210 observaciones entre 2009-01-01 y 2026-06-01; transformacion sugerida=sin_transformacion, d=1, D=0.
- Frontera 01 La Aurora: 210 observaciones entre 2009-01-01 y 2026-06-01; transformacion sugerida=sin_transformacion, d=1, D=0.
- Frontera 07 Valle Nuevo: 210 observaciones entre 2009-01-01 y 2026-06-01; transformacion sugerida=sin_transformacion, d=1, D=0.
- Frontera 09 San Cristóbal: 210 observaciones entre 2009-01-01 y 2026-06-01; transformacion sugerida=boxcox_aproximada, d=1, D=0.
- Vía Aérea: 210 observaciones entre 2009-01-01 y 2026-06-01; transformacion sugerida=sin_transformacion, d=1, D=0.
- Vía Terrestre: 210 observaciones entre 2009-01-01 y 2026-06-01; transformacion sugerida=sin_transformacion, d=1, D=0.
- Vía Marítima: 210 

## 3. Verificar salidas

In [3]:
import pandas as pd

top3 = pd.read_csv(L.TABLAS_DIR / "top_3_fronteras.csv")
est = pd.read_csv(L.TABLAS_DIR / "resultados_estacionariedad.csv")
display(top3)
display(est)

print("Series train/test:")
for p in sorted(L.SERIES_DIR.glob("*.csv")):
    print(" -", p.name)

print("\n--- manifest.md ---\n")
print((L.OUTPUT_DIR / "manifest.md").read_text(encoding="utf-8"))

,Frontera,viajeros
0,01 La Aurora,1.898998e+07
1,07 Valle Nuevo,1.014305e+07
2,09 San Cristóbal,4.183632e+06


,serie,adf_nivel,pvalue_nivel,adf_diff1,pvalue_diff1,d_sugerido,D_sugerido,transformacion,comentario
0,Total mensual,-2.364366,0.152047,-2.987484,3.607991e-02,1,0,sin_transformacion,"ADF nivel=0.15204729499110803, ADF diff1=0.036..."
1,Frontera 01 La Aurora,-2.369729,0.150463,-4.075173,1.064202e-03,1,0,sin_transformacion,"ADF nivel=0.15046311002678492, ADF diff1=0.001..."
2,Frontera 07 Valle Nuevo,-1.831822,0.364781,-3.536127,7.106327e-03,1,0,sin_transformacion,"ADF nivel=0.3647811837972317, ADF diff1=0.0071..."
3,Frontera 09 San Cristóbal,-1.393420,0.585426,-4.075853,1.061465e-03,1,0,boxcox_aproximada,"ADF nivel=0.5854263381023901, ADF diff1=0.0010..."
4,Vía Aérea,-2.367258,0.151192,-4.071047,1.080932e-03,1,0,sin_transformacion,"ADF nivel=0.1511916605938049, ADF diff1=0.0010..."
5,Vía Terrestre,-2.117165,0.237669,-3.326346,1.374112e-02,1,0,sin_transformacion,"ADF nivel=0.23766887868820508, ADF diff1=0.013..."
6,Vía Marítima,0.723740,0.990300,-8.731208,3.188004e-14,1,0,boxcox_aproximada,"ADF nivel=0.9902999277180539, ADF diff1=3.1880..."


Series train/test:
 - frontera_01_la_aurora_test.csv
 - frontera_01_la_aurora_train.csv
 - frontera_07_valle_nuevo_test.csv
 - frontera_07_valle_nuevo_train.csv
 - frontera_09_san_cristobal_test.csv
 - frontera_09_san_cristobal_train.csv
 - total_mensual_test.csv
 - total_mensual_train.csv
 - via_aerea_test.csv
 - via_aerea_train.csv
 - via_maritima_test.csv
 - via_maritima_train.csv
 - via_terrestre_test.csv
 - via_terrestre_train.csv

--- manifest.md ---

# Manifest Parte 1

## Series exactas
- Serie: Total mensual
  - Train/Test: series/total_mensual_train.csv, series/total_mensual_test.csv
  - Transformacion sugerida: sin_transformacion
  - d sugerido: 1
  - D sugerido: 0
  - Observaciones: ADF nivel=0.15204729499110803, ADF diff1=0.03607990560556702, ADF diff estacional=0.8245843452333101, ADF diff1+est=1.3232586364652523e-06. Aditiva y multiplicativa generadas. Descomposiciones: descomposicion_total_mensual_aditiva.png; descomposicion_total_mensual_multiplicativa.png.
- Serie: Fr